# Electronic Sales Analysis

A reproducible customer and revenue analysis using the [Electronic Sales Sep 2023–Sep 2024 dataset on Kaggle](https://www.kaggle.com/datasets/cameronseamons/electronic-sales-sep2023-sep2024).

Only completed orders are included. Each source row is treated as one order because the dataset has no separate order ID. Total spend is `Total Price + Add-on Total`; the source does not specify a currency. A customer cohort is based on the first valid completed order observed in this file.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', font='DejaVu Sans')


In [ ]:
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), Path.cwd().parent)
     if (folder / 'notebooks' / 'electronic_sales_analysis.ipynb').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the cloned project directory.')

CSV_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Electronic_sales_Sep2023-Sep2024.csv'
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f'Download the source CSV from Kaggle and save it at {CSV_PATH}. '
        'The dataset is not bundled with this repository.'
    )

raw = pd.read_csv(CSV_PATH, dtype={'Customer ID': 'string'})
print(f'Raw data: {len(raw):,} rows, {raw.shape[1]} columns')
display(raw.head())
display(raw.isna().sum().rename('missing').to_frame())
display(raw['Order Status'].value_counts().rename('orders').to_frame())
print('Total Price = Unit Price × Quantity:', np.allclose(
    raw['Total Price'], raw['Unit Price'] * raw['Quantity'], equal_nan=True))


## Data cleaning


In [ ]:
COLUMNS = {
    'Customer ID': 'customer_id', 'Payment Method': 'payment_method',
    'Total Price': 'product_total', 'Add-on Total': 'addons_total',
    'Unit Price': 'unit_price', 'Quantity': 'quantity',
    'Purchase Date': 'purchase_date', 'Shipping Type': 'shipping_method',
    'Product Type': 'product_type', 'Order Status': 'order_status',
}

def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the source table without modifying the input DataFrame."""
    missing = sorted(set(COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}")
    d = df.rename(columns=COLUMNS).reindex(columns=list(COLUMNS.values())).copy()
    for col in ['customer_id', 'payment_method', 'shipping_method',
                'product_type', 'order_status']:
        d[col] = d[col].astype('string').str.strip().replace('', pd.NA)
    # Normalize payment method spelling before counting frequencies.
    d['payment_method'] = d['payment_method'].replace({'Paypal': 'PayPal'})
    d['purchase_date'] = pd.to_datetime(d['purchase_date'], format='%Y-%m-%d', errors='coerce')
    addons_text = d['addons_total'].astype('string').str.strip()
    addons_missing = addons_text.isna() | addons_text.eq('')
    for col in ['product_total', 'addons_total', 'unit_price', 'quantity']:
        d[col] = pd.to_numeric(d[col], errors='coerce').astype(float)
    # Replace only genuinely missing add-on totals with zero.
    d.loc[addons_missing, 'addons_total'] = 0.0
    # Exclude missing IDs, dates, or prices; non-finite values; non-positive prices or quantities; fractional quantities; and negative add-on totals.
    valid = (
        d['customer_id'].notna() & d['purchase_date'].notna()
        & np.isfinite(d[['product_total', 'addons_total', 'unit_price', 'quantity']]).all(axis=1)
        & d['product_total'].gt(0) & d['unit_price'].gt(0)
        & d['quantity'].gt(0) & d['quantity'].mod(1).eq(0)
        & d['addons_total'].ge(0)
    )
    completed = d['order_status'].str.casefold().eq('completed').fillna(False)
    audit = {
        'source rows': len(d),
        'not completed (including unknown status)': int((~completed).sum()),
        'invalid completed rows': int((completed & ~valid).sum()),
        'missing add-on totals replaced with 0 before filtering': int(addons_missing.sum()),
    }
    d = d.loc[valid & completed].copy()
    d[['shipping_method', 'product_type']] = d[['shipping_method', 'product_type']].fillna('Unknown')
    d['revenue'] = d['product_total'] + d['addons_total']
    d['month'] = d['purchase_date'].dt.to_period('M')
    audit['retained rows'] = len(d)
    d.attrs['cleaning_report'] = audit
    return d.reset_index(drop=True)

df = clean(raw)
display(pd.Series(df.attrs['cleaning_report'], name='rows').to_frame())
print('Date range after cleaning:', df['purchase_date'].min(), '—', df['purchase_date'].max())


## Task 1: Top 10 customers


In [ ]:
def customer_summary(df: pd.DataFrame) -> pd.DataFrame:
    totals = df.groupby('customer_id', as_index=False).agg(
        total_spent=('revenue', 'sum'),
        addons_spent=('addons_total', 'sum'),
        orders_count=('customer_id', 'size'),
    )
    # Missing payment methods do not count toward frequencies; use Unknown when none are available.
    frequencies = (df.dropna(subset=['payment_method'])
        .groupby(['customer_id', 'payment_method']).size().rename('frequency').reset_index())
    # Break ties by the first method alphabetically after normalizing PayPal spelling.
    preferred = (frequencies.sort_values(
        ['customer_id', 'frequency', 'payment_method'], ascending=[True, False, True])
        .drop_duplicates('customer_id')
        .rename(columns={'payment_method': 'preferred_payment'}))
    result = totals.merge(preferred[['customer_id', 'preferred_payment']], on='customer_id', how='left')
    result['preferred_payment'] = result['preferred_payment'].fillna('Unknown')
    return (result[['customer_id', 'preferred_payment', 'total_spent', 'addons_spent', 'orders_count']]
        .sort_values(['total_spent', 'customer_id'], ascending=[False, True]).reset_index(drop=True))

customers = customer_summary(df)
display(customers.head(10))


## Task 2: Revenue and cohort metrics


In [ ]:
def build_metrics(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if df.empty:
        return {
            'revenue_by_shipping': pd.DataFrame(columns=['shipping_method', 'revenue']),
            'revenue_by_product': pd.DataFrame(columns=['product_type', 'revenue']),
            'addons_by_month': pd.DataFrame(columns=['month', 'addons_revenue']),
            'addons_by_quarter': pd.DataFrame(columns=['quarter', 'addons_revenue']),
            'cohort_revenue': pd.DataFrame(index=pd.PeriodIndex([], freq='M', name='cohort'), columns=pd.Index(range(4), name='age'), dtype=float),
            'cohort_customers': pd.DataFrame(index=pd.PeriodIndex([], freq='M', name='cohort'), columns=pd.Index(range(4), name='age'), dtype=float),
        }
    shipping = (df.groupby('shipping_method', as_index=False)['revenue'].sum()
        .sort_values('revenue', ascending=False).reset_index(drop=True))
    products = (df.groupby('product_type', as_index=False)['revenue'].sum()
        .sort_values('revenue', ascending=False).reset_index(drop=True))
    # Fill months without add-on sales with zero; quarters follow calendar years.
    months = pd.period_range(df['month'].min(), df['month'].max(), freq='M', name='month')
    monthly = (df.groupby('month')['addons_total'].sum().reindex(months, fill_value=0)
        .rename('addons_revenue').reset_index())
    quarterly = (monthly.assign(quarter=monthly['month'].dt.asfreq('Q'))
        .groupby('quarter', as_index=False)['addons_revenue'].sum())

    d = df.copy()
    d['cohort'] = d.groupby('customer_id')['month'].transform('min')
    d['age'] = d['month'].astype('int64') - d['cohort'].astype('int64')
    grouped = d.loc[d['age'].between(0, 3)].groupby(['cohort', 'age']).agg(
        revenue=('revenue', 'sum'), customers=('customer_id', 'nunique'))
    cohorts = pd.PeriodIndex(d['cohort'].unique(), freq='M', name='cohort').sort_values()
    ages = pd.Index(range(4), name='age')
    # Zeros represent observed months without orders; NaN marks future months.
    observed = pd.DataFrame(
        cohorts.asi8[:, None] + np.arange(4)[None, :] <= d['month'].max().ordinal,
        index=cohorts, columns=ages)
    revenue = (grouped['revenue'].unstack('age').reindex(index=cohorts, columns=ages)
        .fillna(0).where(observed))
    buyers = (grouped['customers'].unstack('age').reindex(index=cohorts, columns=ages)
        .fillna(0).where(observed).astype('Int64'))
    return dict(revenue_by_shipping=shipping, revenue_by_product=products,
        addons_by_month=monthly, addons_by_quarter=quarterly,
        cohort_revenue=revenue, cohort_customers=buyers)

metrics = build_metrics(df)
for name, table in metrics.items():  # Iterate over the six result tables, not over data rows.
    display(Markdown(f'### `{name}`'))
    display(table)


## Charts and interpretation

Zeros in a cohort matrix indicate observed months with no orders; blank cells are future months outside the observation window. The first and last calendar months may be incomplete.


In [ ]:
def plot_metrics(metrics):
    if metrics['revenue_by_shipping'].empty:
        print('No valid rows remain after cleaning; charts are skipped.')
        return

    shipping = metrics['revenue_by_shipping']
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(shipping['shipping_method'], shipping['revenue'] / 1e6, color='#3478A3')
    ax.set(title='Revenue by shipping method', xlabel='Shipping method', ylabel='Revenue, millions of units')
    fig.tight_layout()
    plt.show()

    products = metrics['revenue_by_product']
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(products['product_type'], products['revenue'] / 1e6, color='#278579')
    ax.set(title='Revenue by product type', xlabel='Product type', ylabel='Revenue, millions of units')
    fig.tight_layout()
    plt.show()

    monthly = metrics['addons_by_month']
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(monthly['month'].astype(str), monthly['addons_revenue'] / 1e3, marker='o', color='#AB642C')
    ax.set(title='Monthly revenue from add-ons', xlabel='Month', ylabel='Add-on revenue, thousands of units')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(metrics['cohort_revenue'] / 1e3, annot=True, fmt='.1f', cmap='Blues',
        linewidths=.5, ax=ax, cbar_kws={'label': 'Revenue, thousands of units'})
    ax.set(title='Cohort revenue: first four months', xlabel='Months since first order', ylabel='First-order month')
    ax.tick_params(axis='y', rotation=0)
    fig.tight_layout()
    plt.show()

plot_metrics(metrics)


## Edge-case checks


In [ ]:
sample = pd.DataFrame({
    'Customer ID': ['A', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K'],
    'Payment Method': ['Cash', 'Credit Card', None, 'Cash', 'Cash', 'Cash', 'Cash', 'Cash', 'Cash', 'Cash', 'Cash', 'Cash'],
    'Total Price': [100, 100, 50, -1, 10, None, 20, 30, 40, 10, 10, 10],
    'Add-on Total': [10, 0, None, 0, 0, 0, 0, 0, -5, 'bad', 0, 0],
    'Unit Price': [100, 100, 50, 1, 0, 10, 20, 30, 40, 10, 10, 10],
    'Quantity': [1] * 12,
    'Purchase Date': ['2024-01-10', '2024-03-10', '2024-03-11'] + ['2024-02-01'] * 8 + ['bad-date'],
    'Shipping Type': ['Standard'] * 12,
    'Product Type': ['Laptop'] * 12,
    'Order Status': ['Completed'] * 6 + ['Cancelled'] + ['Completed'] * 5,
})
sample.loc[7, 'Customer ID'] = None
sample.loc[10, 'Total Price'] = 0
test_df = clean(sample)
assert len(test_df) == 3
test_customers = customer_summary(test_df).set_index('customer_id')
assert test_customers.loc['A', 'preferred_payment'] == 'Cash'
assert test_customers.loc['A', 'total_spent'] == 210
assert test_customers.loc['B', 'orders_count'] == 1
assert test_customers.loc['B', 'addons_spent'] == 0
assert test_customers.loc['B', 'preferred_payment'] == 'Unknown'
single = sample.iloc[[0]].copy()
assert customer_summary(clean(single)).iloc[0]['preferred_payment'] == 'Cash'
test_metrics = build_metrics(test_df)
assert test_metrics['addons_by_month'].set_index('month').loc[pd.Period('2024-02'), 'addons_revenue'] == 0
assert test_metrics['cohort_revenue'].loc[pd.Period('2024-01'), 1] == 0
assert test_metrics['cohort_revenue'].loc[pd.Period('2024-01'), 2] == 100
assert pd.isna(test_metrics['cohort_revenue'].loc[pd.Period('2024-03'), 1])
assert test_metrics['cohort_customers'].loc[pd.Period('2024-01'), 0] == 1
assert all(t.empty for t in build_metrics(clean(sample.iloc[0:0])).values())
assert all(t.empty for t in build_metrics(clean(sample.iloc[[3]])).values())
assert customer_summary(clean(sample.iloc[0:0])).empty
assert raw.shape[1] == 16  # clean must preserve the input schema.
assert customers['orders_count'].sum() == len(df)
assert np.isclose(customers['total_spent'].sum(), df['revenue'].sum())
assert np.isclose(metrics['revenue_by_shipping']['revenue'].sum(), df['revenue'].sum())
assert np.isclose(metrics['revenue_by_product']['revenue'].sum(), df['revenue'].sum())
assert np.isclose(metrics['addons_by_month']['addons_revenue'].sum(), df['addons_total'].sum())
assert np.isclose(metrics['addons_by_quarter']['addons_revenue'].sum(), df['addons_total'].sum())
assert metrics['cohort_customers'][0].sum() == df['customer_id'].nunique()
print('All checks passed.')


## Summary


In [ ]:
if not df.empty:
    shipping_leader = metrics['revenue_by_shipping'].iloc[0]
    product_leader = metrics['revenue_by_product'].iloc[0]
    monthly = metrics['addons_by_month']
    peak = monthly.loc[monthly['addons_revenue'].idxmax()]
    total = df['revenue'].sum()
    display(Markdown(
        f"- Valid completed orders: **{len(df):,}**, customers: **{df['customer_id'].nunique():,}**. "
        f"Total revenue: **{total:,.2f}** units, including add-ons: **{df['addons_total'].sum():,.2f}**.\n"
        f"- Top shipping method: **{shipping_leader['shipping_method']}**: "
        f"{shipping_leader['revenue'] / total:.1%} of revenue.\n"
        f"- Top product type: **{product_leader['product_type']}**: "
        f"{product_leader['revenue'] / total:.1%} of revenue.\n"
        f"- Peak monthly add-on revenue: **{peak['month']}**: "
        f"**{peak['addons_revenue']:,.2f}** units\n"
        f"- Total cohort revenue in month 0: "
        f"**{metrics['cohort_revenue'][0].sum():,.2f}** units.\n"
        "Revenue by cohort age is not a direct retention comparison because later cohorts have a shorter observation window.\n"
        f"- Cleaned data period: **{df['purchase_date'].min():%Y-%m-%d}–"
        f"{df['purchase_date'].max():%Y-%m-%d}**. The first and last months may be incomplete; "
        "a decline in those months should not be interpreted as seasonality."
    ))
else:
    print('No valid rows remain after cleaning; no conclusions are generated.')
